In [0]:
source_path='/Volumes/financial_guard/source/fraud_watchlist/source_data/'

In [0]:
dbutils.fs.ls(source_path)

[FileInfo(path='dbfs:/Volumes/financial_guard/source/fraud_watchlist/source_data/fraud_watchlist_20261113_131148_487670_0.json', name='fraud_watchlist_20261113_131148_487670_0.json', size=365, modificationTime=1794575509000),
 FileInfo(path='dbfs:/Volumes/financial_guard/source/fraud_watchlist/source_data/fraud_watchlist_20261113_131154_153649_1.json', name='fraud_watchlist_20261113_131154_153649_1.json', size=379, modificationTime=1794575515000),
 FileInfo(path='dbfs:/Volumes/financial_guard/source/fraud_watchlist/source_data/fraud_watchlist_20261113_131159_632041_2.json', name='fraud_watchlist_20261113_131159_632041_2.json', size=380, modificationTime=1794575520000),
 FileInfo(path='dbfs:/Volumes/financial_guard/source/fraud_watchlist/source_data/fraud_watchlist_20261113_131205_187193_3.json', name='fraud_watchlist_20261113_131205_187193_3.json', size=383, modificationTime=1794575526000)]

In [0]:
input_stream = (spark.readStream
                .format("cloudfiles")
                .option("cloudFiles.format", "json")
                .option("cloudFiles.schemaLocation", "/Volumes/financial_guard/source/fraud_watchlist/schema/")
                .option("cloudfiles.inferColumnTypes", "true")
                .load(source_path)
)

In [0]:
from pyspark.sql import functions as F
transformed_df = input_stream.select(
    "*",
    F.col("_metadata.file_path").alias("file_path"),
    F.current_timestamp().alias("ingested_timestamp")
)

In [0]:
streaming_query = (transformed_df.writeStream.format("delta")
.outputMode("Append")
.option("checkpointLocation","/Volumes/financial_guard/source/fraud_watchlist/checkpoint/")
.trigger(availableNow=True)
.toTable("financial_guard.bronze.fraud_watchlist_batch_test")
)

In [0]:
%sql
select * from financial_guard.bronze.fraud_watchlist_batch_test

action,city,country,effective_from,entity_id,reason_code,reason_description,reported_by,reported_source,risk_level,status,watch_type,watchlist_id,_rescued_data,file_path,ingested_timestamp
ADD,Chennai,India,18-Jun-2026 09:04:00,4717327980595032,TEST_TRANSACTION,Auto-generated fraud watchlist entry,AML Team,External Feed,high,ACTIVE,CARD,wl000004,null,/Volumes/financial_guard/source/fraud_watchlist/source_data/fraud_watchlist_20261113_131205_187193_3.json,2026-11-13T13:58:50.568Z
ADD,Delhi,India,18-Jun-2026 09:02:00,5329839542961842,TEST_TRANSACTION,Auto-generated fraud watchlist entry,AML Team,External Feed,low,ACTIVE,CARD,wl000003,null,/Volumes/financial_guard/source/fraud_watchlist/source_data/fraud_watchlist_20261113_131159_632041_2.json,2026-11-13T13:58:50.568Z
ADD,Pune,India,18-Jun-2026 09:01:00,4413337312552355,BLACKLISTED_MERCHANT,Auto-generated fraud watchlist entry,SOC,External Feed,high,ACTIVE,CARD,wl000002,null,/Volumes/financial_guard/source/fraud_watchlist/source_data/fraud_watchlist_20261113_131154_153649_1.json,2026-11-13T13:58:50.568Z
ADD,Hyderabad,India,18-Jun-2026 09:00:00,5008514036965665,PHISHING,Auto-generated fraud watchlist entry,SOC,Manual,high,ACTIVE,CARD,wl000001,null,/Volumes/financial_guard/source/fraud_watchlist/source_data/fraud_watchlist_20261113_131148_487670_0.json,2026-11-13T13:58:50.568Z
